In [580]:
import numpy as np
import pandas as pd
import re
from sklearn.base import BaseEstimator, TransformerMixin


In [581]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')

In [582]:
def sentence_metrics(text):
    sentences = [s.strip() for s in re.split(r'[.!?]+', text) if len(s.strip()) > 0]
    if not sentences:
        return 0, 0, 0
    
    words_per_sentence = [len(s.split()) for s in sentences]
    
    num_sentences = len(sentences)
    avg_words = np.mean(words_per_sentence)
    std_words = np.std(words_per_sentence)
    
    return num_sentences, avg_words, std_words

# humans often alternate long sentences with shorter ones... (I hope so)
sentence_metrics(train['TEXT'][0]), train['LABEL'][0]

((31, np.float64(16.870967741935484), np.float64(7.079010025027744)),
 np.int64(0))

In [583]:
import re
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.base import BaseEstimator, TransformerMixin

class featureExtractor(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass

    @staticmethod
    def __sentence_metrics(text):
        sentences = [s.strip() for s in re.split(r'[.!?]+', text) if len(s.strip()) > 0]
        if not sentences:
            return 0, 0, 0, 0
        words_per_sentence = [len(s.split()) for s in sentences]
        avg   = np.mean(words_per_sentence)
        std   = np.std(words_per_sentence)
        burst = (std - avg) / (std + avg + 1e-9)
        return len(sentences), avg, std, burst

    @staticmethod
    def __hapax_rate(words):
        if not words:
            return 0
        c = Counter(w.lower() for w in words)
        return sum(1 for v in c.values() if v == 1) / len(words)

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        df       = X.copy()
        txt      = df['TEXT'].astype(str)
        words_l  = txt.str.split()
        len_s    = txt.apply(len).replace(0, 1)
        words_s  = words_l.apply(lambda x: len(x) if isinstance(x, list) else 0).replace(0, 1)

        # length
        df['len']       = txt.apply(len)
        df['num_words'] = words_l.apply(lambda x: len(x) if isinstance(x, list) else 0)

        # Textual flags
        df['starts_lower']     = txt.apply(lambda x: x[0].islower() if x else False).astype(int)
        df['contains_genitive']= txt.str.contains("'s").astype(int)

        # Punctuation
        for name, char in [('periods','.'),('commas',','),('dashes','-'),
                            ('question','?'),('exclamation','!'),('semicolon',';'),
                            ('colon',':'),('spaces',' '),('newlines','\n'),('asterisks','*')]:
            df[f'{name}_per_len'] = txt.apply(lambda x: x.count(char)) / len_s

        df['parenthesis_per_len'] = txt.apply(lambda x: x.count('(') + x.count(')')) / len_s
        df['quotes_per_len']      = txt.apply(lambda x: x.count('"') + x.count("'") + x.count('`')) / len_s
        df['uppercase_per_len']   = txt.apply(lambda x: len(re.findall(r'[A-Z]', x))) / len_s
        df['digits_per_len']      = txt.apply(lambda x: len(re.findall(r'\d', x))) / len_s

        # Vocabulary
        df['unique_words_per_words'] = words_l.apply(
            lambda x: len(set(x)) / max(len(x), 1) if isinstance(x, list) else 0
        )
        df['avg_word_length'] = words_l.apply(
            lambda x: np.mean([len(w) for w in x]) if isinstance(x, list) and x else 0
        )
        df['hapax_rate'] = words_l.apply(
            lambda x: self.__hapax_rate(x) if isinstance(x, list) else 0
        )

        # phrase structure
        df['enumeration_num_per_len'] = txt.apply(
            lambda x: len(re.findall(r'\d+\.', x))
        ) / len_s

        metrics = txt.apply(self.__sentence_metrics)
        df['sentences_per_len']      = [m[0] for m in metrics] / len_s
        df['avg_words_per_sentence'] = [m[1] for m in metrics]
        df['std_sentence_length']    = [m[2] for m in metrics]
        df['burstiness']             = [m[3] for m in metrics]
        df['cv_sentence_length']     = df['std_sentence_length'] / (
            df['avg_words_per_sentence'].replace(0, 1)
        )

        return df

    def fit_transform(self, X, y=None, **fit_params):
        return super().fit_transform(X, y, **fit_params)

In [584]:
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(train, test_size=0.25, stratify=train['LABEL'], random_state=42)
X_train, y_train = df_train['TEXT'], df_train['LABEL']
X_test, y_test = df_test['TEXT'], df_test['LABEL']

In [585]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2

class Chi2TextFeatureSelector(BaseEstimator, TransformerMixin):
     """
     This class applies TF-IDF (Term Frequency-Inverse Document Frequency) to text data
     and selects the most important features (words/n-grams) for each class using the 
     Chi-Square (Chi2) statistical test. It uses a One-vs-Rest strategy. (It was also applied in the DSMLL course by me)
     """
     
     def __init__(self, 
                    text_col: str = 'TEXT', 
                    k_per_label: int = 50, 
                    min_df: int = 3,  
                    ngram_range: tuple = (1, 2)):
          
          self.text_col = text_col
          # k_per_label: How many top features to select for each unique class.
          self.k_per_label = k_per_label
          # min_df: Minimum Document Frequency. Ignores words that appear in fewer than 'min_df' documents.
          self.min_df = min_df
          # ngram_range: (1, 2) means we extract single words (unigrams) and two-word phrases (bigrams).
          self.ngram_range = ngram_range
          
          # Attributes that will be learned during the fit()
          self.tfidf_vectorizer_ = None
          self.selected_indices_ = None 
          self.feature_names_ = None
          
     def fit(self, X: pd.DataFrame, y) -> 'Chi2TextFeatureSelector':
          """
          Learns the vocabulary from the text and selects the best features based on the Chi2 test.
          """
          # I absolutely need the labels to compute the Chi-Square statistical test!
          if y is None:
               raise ValueError("Target variable 'y' is required to compute Chi2.")
          
          y_arr = y.values if isinstance(y, pd.Series) else np.array(y)
          
          text_data = X[self.text_col].fillna('').astype(str)
          
          print(f"Fitting TF-IDF (min_df={self.min_df}, ngrams={self.ngram_range})...")
          
          self.tfidf_vectorizer_ = TfidfVectorizer(
               input='content', encoding='utf-8', lowercase=True,
               stop_words=None, # I want to keep stop words because they might be important for classification (e.g., "not", "but", "and") 
               min_df=self.min_df, 
               ngram_range=self.ngram_range
          )
          
         
          # Transform the text into a sparse matrix of TF-IDF scores
          X_tfidf = self.tfidf_vectorizer_.fit_transform(text_data)
          print(f"Selecting top {self.k_per_label} features per label via Chi-Square test...")
          unique_classes = np.unique(y_arr)
          
          feature_to_labels = {}

          # Computing Chi-Square for each class using the "One-vs-Rest" approach
          for label in unique_classes:
               # 1 if it's the current class, 0 otherwise
               y_binary = (y_arr == label).astype(int)
               
               # Compute the Chi2 scores between all TF-IDF features and the binary target
               # The chi2() function returns two arrays: scores and p-values. I only care about the scores.
               chi2_scores, _ = chi2(X_tfidf, y_binary)
               
               # Get the total number of available features in the TF-IDF vocabulary
               n_features = X_tfidf.shape[1]
               
               # Ensure we don't try to select more features than actually exist
               k_safe = min(self.k_per_label, n_features)

               if k_safe > 0:
                    top_k_indices = np.argsort(chi2_scores)[-k_safe:]
                    for idx in top_k_indices:
                         if idx not in feature_to_labels:
                              feature_to_labels[idx] = set()
                              # Record that this specific word index is a strong predictor for the current 'label'
                         feature_to_labels[idx].add(label)

          # Finalizing the selected indices and create descriptive column names
          # Extract all unique indices selected across all classes and sort them
          self.selected_indices_ = sorted(list(feature_to_labels.keys()))
          
          if not self.selected_indices_:
               print("Warning: No features were selected.")
               self.feature_names_ = []
               return self
               
          # Get the actual string words/n-grams from the fitted TF-IDF vocabulary
          raw_feature_names = self.tfidf_vectorizer_.get_feature_names_out()
          self.feature_names_ = []
          
          for idx in self.selected_indices_:
               # Extract the actual word corresponding to the numerical index
               word = raw_feature_names[idx]
               
               # Create a string suffix of the labels that this word helps predict (e.g., "0_3")
               labels_suffix = "_".join(sorted([str(lbl) for lbl in feature_to_labels[idx]]))
               
               # Construct the final descriptive feature name (e.g., "tfidf_apple_L0_3")
               self.feature_names_.append(f"tfidf_{word}_L{labels_suffix}")
               
          print(f"Total unique TF-IDF features selected: {len(self.selected_indices_)}")
          return self
     
     def transform(self, X: pd.DataFrame) -> pd.DataFrame:
          """
          Applies the learned TF-IDF transformation and filters the matrix to keep 
          only the features selected by the Chi2 test during the fit() phase.
          """
          # Check if the model has been fitted properly
          if self.tfidf_vectorizer_ is None:
               raise RuntimeError("The Transformer is not fitted yet. Call fit() before transform().")

          # Create a copy to avoid altering the original dataframe in memory
          df = X.copy()
          text_data = df[self.text_col].fillna('').astype(str)
          
          # Initialize a list of dataframes to concatenate at the end.
          # We start by removing the original raw text column so it doesn't get passed to the ML model.
          dfs_to_concat = [df.drop(columns=[self.text_col], errors='ignore')]
          
          if len(self.selected_indices_) > 0:
               # Transform the new text data using the ENTIRE vocabulary learned during fit()
               X_tfidf_full = self.tfidf_vectorizer_.transform(text_data)
               
               # Feature Selection (Filtering)
               # Slice the sparse matrix: keep ONLY the columns (indices) selected by the Chi2 test
               X_tfidf_sel = X_tfidf_full[:, self.selected_indices_]
               
               # Convert the sparse matrix into a dense Pandas DataFrame
               df_tfidf = pd.DataFrame(
                    X_tfidf_sel.toarray(),         # .toarray() converts the sparse matrix to a dense NumPy array
                    columns=self.feature_names_,   # Apply the descriptive column names we generated in fit()
                    index=df.index                 # Keep the original index to ensure alignment with other features
               )
               
               # Add the new TF-IDF numerical dataframe to our concatenation list
               dfs_to_concat.append(df_tfidf)
               
          # Concatenate horizontally (axis=1) to combine any existing features with the new text features
          final_df = pd.concat(dfs_to_concat, axis=1)
          return final_df

In [586]:
X_train = pd.DataFrame(X_train)

In [587]:
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator, TransformerMixin

class SelectiveScaler(BaseEstimator, TransformerMixin):
    def __init__(self, exclude_cols=['TEXT']):
        self.exclude_cols = exclude_cols
        self.scaler = StandardScaler()
        self.features_to_scale_ = None

    def fit(self, X, y=None):
        # Identify all columns that are NOT text
        self.features_to_scale_ = [c for c in X.columns if c not in self.exclude_cols]
        if self.features_to_scale_:
            self.scaler.fit(X[self.features_to_scale_])
        return self

    def transform(self, X):
        X_scaled = X.copy()
        if self.features_to_scale_:
            # Scale only the numerical features in place
            X_scaled[self.features_to_scale_] = self.scaler.transform(X[self.features_to_scale_])
        return X_scaled

In [588]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline(
     [
          
          ('feature_ext', featureExtractor()),
          ('impChi', Chi2TextFeatureSelector(k_per_label=45, min_df=3, ngram_range=(1, 3))),
          ('ss', StandardScaler())
     ]
)
X_train_pp = pipeline.fit_transform(pd.DataFrame(X_train), y_train)
X_test_pp = pipeline.transform(pd.DataFrame(X_test))

Fitting TF-IDF (min_df=3, ngrams=(1, 3))...
Selecting top 45 features per label via Chi-Square test...
Total unique TF-IDF features selected: 257


In [589]:
# robust one
params_31 = {'objective':'multiclass',
    'num_class':6,
    'class_weight':'balanced',
    'boosting_type':'gbdt',
    'random_state':42,
    'n_jobs':-1,
    'verbosity':-1,'n_estimators': 730, 'learning_rate': 0.010407498940906855, 'max_depth': 5, 'num_leaves': 54, 'min_child_samples': 37, 'min_split_gain': 0.06963977833346169, 'reg_alpha': 0.21148540123122064, 'reg_lambda': 0.030761881469967566, 'colsample_bytree': 0.3518507408719965, 'subsample': 0.9676247445281039, 'subsample_freq': 6}

# gamble one
params_26 = {'objective':'multiclass',
    'num_class':6,
    'class_weight':'balanced',
    'boosting_type':'gbdt',
    'random_state':42,
    'n_jobs':-1,
    'verbosity':-1,'n_estimators': 400, 'learning_rate': 0.02349778714378209, 'max_depth': 12, 'num_leaves': 248, 'min_child_samples': 67, 'min_split_gain': 0.044000000000000004, 'reg_alpha': 0.2150868646977478, 'reg_lambda': 1.2045035383792522, 'colsample_bytree': 0.3370000000000001, 'subsample': 0.8870000000000001, 'subsample_freq': 4}

# robust
params_8 = {'objective':'multiclass',
    'num_class':6,
    'class_weight':'balanced',
    'boosting_type':'gbdt',
    'random_state':42,
    'n_jobs':-1,
    'verbosity':-1,'n_estimators': 922, 'learning_rate': 0.00531016167886792, 'max_depth': 8, 'num_leaves': 50, 'min_child_samples': 46, 'min_split_gain': 0.09890451535072747, 'reg_alpha': 0.013804574251952207, 'reg_lambda': 0.2663921229953439, 'colsample_bytree': 0.24448771356145826, 'subsample': 0.8606974769886846, 'subsample_freq': 5}

In [590]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier


# Training the two best promising models
print('training model 1(trial 31)')
model_31 = LGBMClassifier(**params_31).fit(X_train_pp, y_train)

print("training model 2 (trial 8)")
model_8 = LGBMClassifier(**params_8).fit(X_train_pp, y_train)

# extracting probabilities
probs_31 = model_31.predict_proba(X_test_pp)
probs_8  = model_8.predict_proba(X_test_pp)

# trying soft voting just for fun... does not work
blended_probs = (probs_31 + probs_8 ) / 2.0

final_predictions = np.argmax(blended_probs, axis=1)

from sklearn.metrics import classification_report, f1_score
print(f"\nF1 Macro Score: {f1_score(y_test, final_predictions, average='macro'):.4f}")
print(classification_report(y_test, final_predictions, digits=4))

training model 1(trial 31)
training model 2 (trial 8)

F1 Macro Score: 0.9548
              precision    recall  f1-score   support

           0     1.0000    0.9974    0.9987       380
           1     0.7600    0.9500    0.8444        20
           2     0.9714    0.8500    0.9067        40
           3     1.0000    1.0000    1.0000        20
           4     0.9836    1.0000    0.9917        60
           5     0.9875    0.9875    0.9875        80

    accuracy                         0.9850       600
   macro avg     0.9504    0.9641    0.9548       600
weighted avg     0.9868    0.9850    0.9853       600



c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score, classification_report

estimators = [
    ('lgbm_31', CalibratedClassifierCV(
        LGBMClassifier(**params_31),
        method='isotonic', cv=5
    )),
    ('lgbm_8', CalibratedClassifierCV(
        LGBMClassifier(**params_8),
        method='isotonic', cv=5
    )),
]

meta = LogisticRegression(
    class_weight='balanced',
    C=0.1,
    max_iter=1000,
    random_state=42
)

stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=meta,
    cv=5,
    stack_method='predict_proba',
    passthrough=False,
    n_jobs=-1
)

stacking_clf.fit(X_train_pp, y_train)
y_pred = stacking_clf.predict(X_test_pp)
print(f"Stacking F1 Macro: {f1_score(y_test, y_pred, average='macro'):.4f}")
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
full_pipeline = Pipeline(
     [
          ('preprocessing_ext', pipeline),
          ('model', stacking_clf)
     ]
)

X,y = train['TEXT'], train['LABEL']
full_pipeline.fit(pd.DataFrame(X), y)
test_preds = full_pipeline.predict(pd.DataFrame(test['TEXT']))
submission = pd.DataFrame({'ID': np.arange(test.shape[0]), 'LABEL': test_preds})
submission.to_csv('submission.csv', index=False)

Fitting TF-IDF (min_df=3, ngrams=(1, 3))...
Selecting top 45 features per label via Chi-Square test...
Total unique TF-IDF features selected: 256


c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\gianb\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: U